# YOLOv8 PPE Detection — Fine-Tuning
**Dataset:** Roboflow Construction Safety (4 classes)
**Runtime:** GPU (T4) — Go to Runtime → Change runtime type → GPU
**Time:** ~25-30 minutes

In [ ]:
# Step 1: Install dependencies
!pip install ultralytics roboflow -q

In [ ]:
# Step 2: Download PPE dataset from Roboflow (free, no API key needed for public datasets)
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")  # Get free key at https://app.roboflow.com/
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
version = project.version(30)
dataset = version.download("yolov8")
print(f"Dataset downloaded to: {dataset.location}")

In [ ]:
# Step 3: Train YOLOv8n on PPE dataset
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # Load pretrained nano model

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    optimizer='SGD',
    lr0=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    project='runs/train',
    name='ppe_yolov8n',
    save=True,
    plots=True,
)
print("Training complete!")

In [ ]:
# Step 4: Evaluate
model = YOLO('runs/train/ppe_yolov8n/weights/best.pt')
metrics = model.val()
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

In [ ]:
# Step 5: Test on a sample image
import glob
test_images = glob.glob(f"{dataset.location}/test/images/*")[:3]
results = model.predict(test_images, save=True, conf=0.4)
print("Predictions saved to runs/detect/")

In [ ]:
# Step 6: Download the trained model to your local machine
from google.colab import files
files.download('runs/train/ppe_yolov8n/weights/best.pt')
print("Download best.pt and place it in your YOLO_project/ folder")